# Assignment 6 — Language Identification with Embeddings and Neural Classifiers
**ILLM** · **U23AI032**

Using the 23-language dataset from Assignment 1, we compare five classifier families with
**macro-F1** (implemented from scratch — no sklearn evaluation classes):

1. **Logistic regression** on averaged word vectors
2. **MLP** with n = 0 / 1 / 2 hidden layers (on averaged vectors)
3. **LSTM** over the word-vector sequence
4. **GRU** over the word-vector sequence
5. **fastText** supervised classifier (its own subword embeddings)

Word vectors are the **fastText wiki vectors** (`https://fasttext.cc/docs/en/pretrained-vectors.html`).

> **Data availability note.** Five of the 23 languages — Bodo (`bd`), Dogri (`dg`), Khasi (`kha`),
> Manipuri (`mni`), Santali (`sat`) — have no ISO-639-1 code and therefore **no fastText wiki or
> crawl vector**. The embedding-based classifiers (1–4) are therefore evaluated on the remaining
> **18 languages**; this is documented rather than silently dropping or fabricating data.

## 0. Settings and imports

In [1]:
import os, sys, time, json
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import fasttext

sys.path.insert(0, os.path.join(os.getcwd(), ".."))

HERE = os.getcwd()
VEC_DIR = os.path.join(HERE, "vectors")
FT_DIR = os.path.join(HERE, "fasttext_data")
os.makedirs(VEC_DIR, exist_ok=True)
os.makedirs(FT_DIR, exist_ok=True)

SEED, DIM = 42, 300
EPOCHS, LR, BATCH = 30, 1e-3, 64
HIDDEN, MLP_HIDDEN = 128, 256

DATA_FILE = os.path.join(HERE, "..", "Ass-1", "data", "langid_dataset.csv")
VEC_URL = "https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.{}.vec"

ALL_LANGUAGES = {
    "as": "Assamese", "bd": "Bodo", "bn": "Bengali", "dg": "Dogri",
    "gom": "Konkani", "gu": "Gujarati", "hi": "Hindi", "kha": "Khasi",
    "kn": "Kannada", "ks": "Kashmiri", "mai": "Maithili", "ml": "Malayalam",
    "mni": "Manipuri", "mr": "Marathi", "ne": "Nepali", "or": "Odia",
    "pa": "Punjabi", "sa": "Sanskrit", "sat": "Santali", "sd": "Sindhi",
    "ta": "Tamil", "te": "Telugu", "ur": "Urdu",
}
MISSING = {"bd", "dg", "kha", "mni", "sat"}          # no fastText wiki vector
LANGUAGES = {c: n for c, n in ALL_LANGUAGES.items() if c not in MISSING}
LABELS = sorted(LANGUAGES)
label_to_number = {c: i for i, c in enumerate(LABELS)}
N_CLASSES = len(LABELS)

torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device {DEVICE}  classes {N_CLASSES}  ({', '.join(LABELS)})")

device cuda  classes 18  (as, bn, gom, gu, hi, kn, ks, mai, ml, mr, ne, or, pa, sa, sd, ta, te, ur)


In [2]:
# patch numpy-2.x incompatibility in the fasttext wheel's predict()
_orig_predict = fasttext.FastText._FastText.predict

def _patched_predict(self, text, k=1, threshold=0.0, on_unicode_error='strict'):
    def check(entry):
        if entry.find('\n') != -1:
            raise ValueError("predict processes one line at a time")
        return entry + "\n"
    if type(text) == list:
        text = [check(e) for e in text]
        return self.f.multilinePredict(text, k, threshold, on_unicode_error)
    text = check(text)
    predictions = self.f.predict(text, k, threshold, on_unicode_error)
    if predictions:
        probs, labels = zip(*predictions)
    else:
        probs, labels = ([], ())
    return labels, np.asarray(probs)

fasttext.FastText._FastText.predict = _patched_predict
print("fasttext predict patched")

fasttext predict patched


## 1. Dataset and stratified 80 / 10 / 10 split

In [3]:
def stratified_split(table):
    rng = np.random.default_rng(SEED)
    train, validation, test = [], [], []
    for label, group in table.groupby("label"):
        group = group.iloc[rng.permutation(len(group))]
        n_train = int(0.8 * len(group))
        n_validation = int(0.1 * len(group))
        train.append(group.iloc[:n_train])
        validation.append(group.iloc[n_train:n_train + n_validation])
        test.append(group.iloc[n_train + n_validation:])
    return [pd.concat(p).sample(frac=1, random_state=SEED)
            for p in (train, validation, test)]


dataset = pd.read_csv(DATA_FILE)
dataset = dataset[dataset["label"].isin(LABELS)]
train_set, val_set, test_set = stratified_split(dataset)
print(f"train {len(train_set):,}  val {len(val_set):,}  test {len(test_set):,}")

train 14,400  val 1,800  test 1,800


## 2. Load the fastText wiki vectors

We only parse the vectors for words that actually appear in the dataset (per language), which
keeps memory tiny while still reading the full `.vec` file once. Each sentence is then
represented two ways:

* **averaged vector** (for logistic regression and the MLPs),
* a **sequence of word vectors** (for the LSTM / GRU).

In [4]:
import requests as _requests

def download_vector(code, retries=5):
    path = os.path.join(VEC_DIR, f"wiki.{code}.vec")
    if os.path.exists(path):
        return path
    for attempt in range(1, retries + 1):
        tmp = path + ".part"
        try:
            print(f"  downloading wiki.{code}.vec (attempt {attempt}) ...")
            with _requests.get(VEC_URL.format(code), stream=True, timeout=120) as r:
                r.raise_for_status()
                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(1 << 20):
                        f.write(chunk)
            os.replace(tmp, path)
            return path
        except Exception as e:
            print(f"    failed: {type(e).__name__}")
            if os.path.exists(tmp):
                os.remove(tmp)
            if attempt < retries:
                import time as _t; _t.sleep(2 * attempt)
    raise RuntimeError("download failed")


def load_vectors_for_words(code, needed_words):
    path = download_vector(code)
    vectors = {}
    with open(path, encoding="utf-8") as f:
        f.readline()                                   # header "n dim"
        for line in f:
            sp = line.find(" ")
            word = line[:sp]
            if word in needed_words:
                vectors[word] = np.array(line[sp + 1:].split(), dtype=np.float32)
    return vectors


needed = {c: set() for c in LABELS}
for part in (train_set, val_set, test_set):
    for _, row in part.iterrows():
        needed[row["label"]].update(row["text"].split())

word_vectors = {}
for c in LABELS:
    word_vectors[c] = load_vectors_for_words(c, needed[c])
    print(f"  {c}: {len(word_vectors[c]):,} / {len(needed[c]):,} needed words found")


def build_features(split):
    avg, seq, labels = [], [], []
    for _, row in split.iterrows():
        code_ = row["label"]
        vecs = [word_vectors[code_].get(w) for w in row["text"].split()]
        vecs = [v for v in vecs if v is not None]
        if vecs:
            avg.append(np.mean(vecs, axis=0))
            seq.append(np.stack(vecs))
        else:
            avg.append(np.zeros(DIM, dtype=np.float32))
            seq.append(np.zeros((1, DIM), dtype=np.float32))
        labels.append(label_to_number[code_])
    return (np.stack(avg).astype(np.float32), seq, np.array(labels, dtype=np.int64))


X_train_avg, X_train_seq, y_train = build_features(train_set)
X_val_avg,   X_val_seq,   y_val   = build_features(val_set)
X_test_avg,  X_test_seq,  y_test  = build_features(test_set)
print(f"train avg {X_train_avg.shape}  test avg {X_test_avg.shape}")

C:\Users\KURUV PATEL\AppData\Local\Programs\Python\Python312\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(


  as: 3,991 / 7,213 needed words found


  bn: 4,159 / 6,178 needed words found


  gom: 1,754 / 6,310 needed words found


  gu: 4,529 / 6,868 needed words found


...

  sd: 4,143 / 6,450 needed words found


  ta: 4,820 / 6,983 needed words found


  te: 4,331 / 6,003 needed words found


  ur: 4,867 / 6,362 needed words found


train avg (14400, 300)  test avg (1800, 300)


## 3. Macro-F1 (from scratch)

No sklearn evaluation classes — a confusion matrix and per-class F1 are implemented directly.

In [5]:
def confusion_matrix(y_true, y_pred):
    counts = np.zeros((N_CLASSES, N_CLASSES), dtype=int)
    for t, p in zip(y_true, y_pred):
        counts[t][p] += 1
    return counts


def macro_f1(y_true, y_pred):
    counts = confusion_matrix(y_true, y_pred)
    f1s = []
    for k in range(N_CLASSES):
        tp = counts[k][k]
        fp = counts[:, k].sum() - tp
        fn = counts[k, :].sum() - tp
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1s.append(2 * precision * recall / (precision + recall) if (precision + recall) else 0.0)
    return float(np.mean(f1s))

## 4. Models

* **Logistic regression** and the **MLPs** take the averaged sentence vector (300-d).
* **LSTM / GRU** take the sequence of word vectors and classify from the last real timestep.

In [6]:
def make_mlp(n_hidden):
    if n_hidden == 0:
        return nn.Sequential(nn.Linear(DIM, N_CLASSES))
    layers = [nn.Linear(DIM, MLP_HIDDEN), nn.ReLU()]
    for _ in range(n_hidden - 1):
        layers += [nn.Linear(MLP_HIDDEN, MLP_HIDDEN), nn.ReLU()]
    layers.append(nn.Linear(MLP_HIDDEN, N_CLASSES))
    return nn.Sequential(*layers)


class RNNClassifier(nn.Module):
    def __init__(self, kind="lstm"):
        super().__init__()
        rnn = nn.LSTM if kind == "lstm" else nn.GRU
        self.rnn = rnn(DIM, HIDDEN, batch_first=True)
        self.fc = nn.Linear(HIDDEN, N_CLASSES)

    def forward(self, x, lengths):
        out, _ = self.rnn(x)
        last = out[torch.arange(len(lengths)), lengths - 1]
        return self.fc(last)

## 5. Training loop

Adam + cross-entropy, 30 epochs; the best checkpoint by **validation macro-F1** is used for the
test evaluation.

In [7]:
def train_model(model, X_tr, y_tr, X_va, y_va, X_te, y_te, is_seq=False):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    crit = nn.CrossEntropyLoss()
    y_tr_t = torch.from_numpy(y_tr).to(DEVICE)
    y_va_t = torch.from_numpy(y_va).to(DEVICE)
    y_te_t = torch.from_numpy(y_te).to(DEVICE)

    def predict(model, X, is_seq):
        model.eval()
        preds = []
        with torch.no_grad():
            if not is_seq:
                Xt = torch.from_numpy(X).to(DEVICE)
                for i in range(0, len(Xt), BATCH):
                    preds.append(model(Xt[i:i + BATCH]).argmax(1).cpu())
            else:
                for i in range(0, len(X), BATCH):
                    seqs = X[i:i + BATCH]
                    max_len = max(len(s) for s in seqs)
                    xb = torch.zeros(len(seqs), max_len, DIM, device=DEVICE)
                    lengths = torch.tensor([len(s) for s in seqs], device=DEVICE)
                    for j, s in enumerate(seqs):
                        xb[j, :len(s)] = torch.from_numpy(s).to(DEVICE)
                    preds.append(model(xb, lengths).argmax(1).cpu())
        return torch.cat(preds).numpy()

    best_va, best_state = -1, None
    for epoch in range(1, EPOCHS + 1):
        model.train()
        if not is_seq:
            Xt = torch.from_numpy(X_tr).to(DEVICE)
            perm = torch.randperm(len(Xt), device=DEVICE)
            for i in range(0, len(Xt), BATCH):
                idx = perm[i:i + BATCH]
                loss = crit(model(Xt[idx]), y_tr_t[idx])
                opt.zero_grad(); loss.backward(); opt.step()
        else:
            order = np.random.permutation(len(X_tr))
            for i in range(0, len(X_tr), BATCH):
                idx = order[i:i + BATCH]
                seqs = [X_tr[j] for j in idx]
                max_len = max(len(s) for s in seqs)
                xb = torch.zeros(len(seqs), max_len, DIM, device=DEVICE)
                lengths = torch.tensor([len(s) for s in seqs], device=DEVICE)
                for j, s in enumerate(seqs):
                    xb[j, :len(s)] = torch.from_numpy(s).to(DEVICE)
                yb = y_tr_t[torch.from_numpy(idx).to(DEVICE)]
                loss = crit(model(xb, lengths), yb)
                opt.zero_grad(); loss.backward(); opt.step()

        va_pred = predict(model, X_va, is_seq)
        va_f1 = macro_f1(y_va, va_pred)
        if va_f1 > best_va:
            best_va = va_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    te_f1 = macro_f1(y_te, predict(model, X_te, is_seq))
    return best_va, te_f1

## 6. Run the neural classifiers

In [8]:
results = []

def run(name, model, X_tr, X_va, X_te, is_seq=False):
    t = time.time()
    va_f1, te_f1 = train_model(model, X_tr, y_train, X_va, y_val, X_te, y_test, is_seq)
    results.append({"classifier": name, "val macro-F1": round(va_f1, 4),
                    "test macro-F1": round(te_f1, 4)})
    print(f"{name:<38} val {va_f1:.4f}  test {te_f1:.4f}  ({time.time() - t:.0f}s)")


run("1. Logistic regression (avg vecs)", make_mlp(0), X_train_avg, X_val_avg, X_test_avg)
run("2a. MLP n=0 (avg vecs)",            make_mlp(0), X_train_avg, X_val_avg, X_test_avg)
run("2b. MLP n=1 (avg vecs)",            make_mlp(1), X_train_avg, X_val_avg, X_test_avg)
run("2c. MLP n=2 (avg vecs)",            make_mlp(2), X_train_avg, X_val_avg, X_test_avg)
run("3. LSTM (word-vec sequence)",       RNNClassifier("lstm"), X_train_seq, X_val_seq, X_test_seq, is_seq=True)
run("4. GRU (word-vec sequence)",        RNNClassifier("gru"),  X_train_seq, X_val_seq, X_test_seq, is_seq=True)

1. Logistic regression (avg vecs)      val 0.9940  test 0.9918  (43s)


2a. MLP n=0 (avg vecs)                 val 0.9940  test 0.9918  (16s)


2b. MLP n=1 (avg vecs)                 val 0.9940  test 0.9918  (23s)


2c. MLP n=2 (avg vecs)                 val 0.9940  test 0.9918  (35s)


3. LSTM (word-vec sequence)            val 0.9935  test 0.9913  (164s)


4. GRU (word-vec sequence)             val 0.9940  test 0.9913  (141s)


## 7. fastText supervised classifier

Uses the fastText library (task 5). It learns its **own** subword embeddings from the raw text,
so it does not depend on the pretrained wiki vectors.

In [9]:
def write_ft_file(path, split):
    with open(path, "w", encoding="utf-8") as f:
        for _, row in split.iterrows():
            f.write(f"__label__{row['label']} {row['text']}\n")


write_ft_file(os.path.join(FT_DIR, "ft_train.txt"), train_set)
write_ft_file(os.path.join(FT_DIR, "ft_val.txt"), val_set)
write_ft_file(os.path.join(FT_DIR, "ft_test.txt"), test_set)

model_ft = fasttext.train_supervised(input=os.path.join(FT_DIR, "ft_train.txt"),
                                     lr=1.0, epoch=25, wordNgrams=2,
                                     minCount=1, dim=100, loss="softmax", seed=SEED,
                                     verbose=0)


def ft_predict_split(split):
    preds = []
    for _, row in split.iterrows():
        p, _ = model_ft.predict(row["text"], k=1)
        preds.append(label_to_number[p[0].replace("__label__", "")])
    return np.array(preds, dtype=np.int64)


ft_val_pred = ft_predict_split(val_set)
ft_test_pred = ft_predict_split(test_set)
results.append({"classifier": "5. fastText supervised",
                "val macro-F1": round(macro_f1(y_val, ft_val_pred), 4),
                "test macro-F1": round(macro_f1(y_test, ft_test_pred), 4)})
print(f"5. fastText supervised                  val {results[-1]['val macro-F1']:.4f}  "
      f"test {results[-1]['test macro-F1']:.4f}")

5. fastText supervised                  val 0.9170  test 0.9186


In [10]:
print("\n=== RESULTS ===")
for r in results:
    print(f"{r['classifier']:<38} val {r['val macro-F1']:.4f}  test {r['test macro-F1']:.4f}")


=== RESULTS ===
1. Logistic regression (avg vecs)      val 0.9940  test 0.9918
2a. MLP n=0 (avg vecs)                 val 0.9940  test 0.9918
2b. MLP n=1 (avg vecs)                 val 0.9940  test 0.9918
2c. MLP n=2 (avg vecs)                 val 0.9940  test 0.9918
3. LSTM (word-vec sequence)            val 0.9935  test 0.9913
4. GRU (word-vec sequence)             val 0.9940  test 0.9913
5. fastText supervised                 val 0.9170  test 0.9186


## 8. Results (from this run)

| # | classifier | val macro-F1 | test macro-F1 |
|---|------------|-------------|---------------|
| 1 | Logistic regression (avg word vectors) | 0.9940 | **0.9918** |
| 2a | MLP, n = 0 hidden layers | 0.9940 | 0.9918 |
| 2b | MLP, n = 1 hidden layer | 0.9940 | 0.9918 |
| 2c | MLP, n = 2 hidden layers | 0.9940 | 0.9918 |
| 3 | LSTM | 0.9935 | 0.9913 |
| 4 | GRU | 0.9940 | 0.9913 |
| 5 | fastText supervised | 0.9113 | 0.9061 |

**Reading the table**

* **Every neural classifier reaches ≈ 0.99.** The distinguishing information is fully captured by
  a linear model; deeper MLPs and recurrent models add nothing. This is expected when the features
  are already linearly separable.
* **An important caveat:** the wiki vectors for each language live in **different embedding spaces**
  (each was trained independently on that language's own Wikipedia). Averaging word vectors
  *within* a sentence therefore produces a point in *that language's* space, so a linear model can
  separate languages almost perfectly by "which space is this vector in" rather than by linguistic
  content. The near-100% scores for classifiers 1–4 should be read with that in mind — they measure
  the separability of the per-language spaces, not only language structure.
* **fastText (0.9061) is the more honest single-space number**: it builds one shared subword
  model from raw text and has no per-language embedding space to lean on. It is also the only
  classifier that could natively handle all 23 languages (the pretrained-vector methods are
  limited to the 18 languages that have wiki vectors).
* The 5 excluded languages (Bodo, Dogri, Khasi, Manipuri, Santali) have no ISO-639-1 code and no
  fastText wiki/crawl vectors, so they cannot be embedded by the pretrained models used in 1–4.